In [ ]:
## SIN OPTUNA

# ============================================================
# PIPELINE COMPLETO SIN FUNCIONES
# ============================================================

# pip install pandas numpy scikit-learn xgboost tensorflow scipy

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.inspection import permutation_importance

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from scipy import sparse
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# CONFIG
# ============================================================
TARGET = "producto_objetivo"

# df = tu dataframe aquí
# ------------------------------------------------------------

# ============================================================
# 1. TARGET
# ============================================================
le = LabelEncoder()
y = le.fit_transform(df[TARGET].astype(str))

X = df.drop(columns=[TARGET])

# ============================================================
# 2. LIMPIEZA
# ============================================================
miss = X.isnull().mean()
X = X.loc[:, miss < 0.95]

num_cols = X.select_dtypes(include=np.number).columns
var = X[num_cols].var()
good_num = var[var > 1e-5].index

X = pd.concat([X[good_num], X.select_dtypes(exclude=np.number)], axis=1)

# ============================================================
# 3. ONE HOT PARA MI
# ============================================================
X_enc = pd.get_dummies(X, drop_first=True)

# ============================================================
# 4. MUTUAL INFORMATION
# ============================================================
mi = mutual_info_classif(X_enc, y)
mi = pd.Series(mi, index=X_enc.columns).sort_values(ascending=False)

top_mi = mi.head(300).index
X_enc = X_enc[top_mi]

# ============================================================
# 5. XGBOOST IMPORTANCE (ROBUSTO)
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

importances = pd.DataFrame(index=X_enc.columns)

for i, (tr, va) in enumerate(skf.split(X_enc, y)):
    model_xgb = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss"
    )
    
    model_xgb.fit(X_enc.iloc[tr], y[tr])
    
    imp = pd.Series(model_xgb.feature_importances_, index=X_enc.columns)
    importances[f"fold_{i}"] = imp

importances["mean"] = importances.mean(axis=1)
importances = importances.sort_values("mean", ascending=False)

top_xgb = importances.head(150).index
X_final = X_enc[top_xgb]

# ============================================================
# 6. SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, stratify=y, random_state=SEED
)

# ============================================================
# 7. ESCALADO
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# 8. CLASS WEIGHT
# ============================================================
weights = compute_class_weight(
    "balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight = {i: w for i, w in enumerate(weights)}

# ============================================================
# 9. RED NEURONAL (con L1 para selección implícita)
# ============================================================
inputs = keras.Input(shape=(X_train.shape[1],))

x = layers.Dense(256, activation="relu",
                 kernel_regularizer=regularizers.l1(1e-4))(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l1(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64, activation="relu")(x)

outputs = layers.Dense(len(np.unique(y)), activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True
    )
]

model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=128,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# 10. PREDICCIONES
# ============================================================
proba = model.predict(X_test)
preds = np.argmax(proba, axis=1)

# ============================================================
# 11. MÉTRICAS
# ============================================================
def recall_at_k(y_true, y_proba, k=3):
    topk = np.argsort(-y_proba, axis=1)[:, :k]
    return np.mean([y_true[i] in topk[i] for i in range(len(y_true))])

def mrr(y_true, y_proba):
    order = np.argsort(-y_proba, axis=1)
    rr = []
    for i in range(len(y_true)):
        pos = np.where(order[i] == y_true[i])[0][0] + 1
        rr.append(1 / pos)
    return np.mean(rr)

print("\n=== RESULTADOS ===")
print("Accuracy:", accuracy_score(y_test, preds))
print("F1:", f1_score(y_test, preds, average="weighted"))
print("LogLoss:", log_loss(y_test, proba))
print("Recall@3:", recall_at_k(y_test, proba, 3))
print("MRR:", mrr(y_test, proba))

# ============================================================
# 12. FEATURE IMPORTANCE (RED NEURONAL)
# Permutation Importance (la forma correcta en NN)
# ============================================================
print("\nCalculando importancia de variables (esto puede tardar)...")

perm = permutation_importance(
    estimator=model,
    X=X_test,
    y=y_test,
    n_repeats=5,
    random_state=SEED,
    scoring="neg_log_loss"
)

nn_importance = pd.Series(
    perm.importances_mean,
    index=top_xgb
).sort_values(ascending=False)

print("\n=== TOP FEATURES NN ===")
print(nn_importance.head(20))

# ============================================================
# 13. RECOMENDACIONES TOP-N
# ============================================================
top_n = 3

classes = le.inverse_transform(np.arange(proba.shape[1]))
order = np.argsort(-proba, axis=1)[:, :top_n]

rows = []
for i in range(len(X_test)):
    row = {}
    for j in range(top_n):
        idx = order[i, j]
        row[f"producto_{j+1}"] = classes[idx]
        row[f"prob_{j+1}"] = float(proba[i, idx])
    rows.append(row)

ranking = pd.DataFrame(rows)

print("\n=== EJEMPLO RECOMENDACIONES ===")
print(ranking.head())

In [ ]:
# CON OPTUNA Y OPTIMIZANDO RECALL@3

# ============================================================
# PIPELINE COMPLETO + OPTUNA OPTIMIZANDO RECALL@3
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import optuna
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TARGET = "producto_objetivo"

# ============================================================
# PREPROCESAMIENTO (igual que antes)
# ============================================================
le = LabelEncoder()
y = le.fit_transform(df[TARGET].astype(str))
X = df.drop(columns=[TARGET])

miss = X.isnull().mean()
X = X.loc[:, miss < 0.95]

num_cols = X.select_dtypes(include=np.number).columns
var = X[num_cols].var()
good_num = var[var > 1e-5].index

X = pd.concat([X[good_num], X.select_dtypes(exclude=np.number)], axis=1)

X_enc = pd.get_dummies(X, drop_first=True)

mi = mutual_info_classif(X_enc, y)
mi = pd.Series(mi, index=X_enc.columns).sort_values(ascending=False)

X_enc = X_enc[mi.head(300).index]

# XGBoost selección
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
importances = pd.DataFrame(index=X_enc.columns)

for i, (tr, va) in enumerate(skf.split(X_enc, y)):
    model_xgb = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss"
    )
    model_xgb.fit(X_enc.iloc[tr], y[tr])
    importances[f"fold_{i}"] = model_xgb.feature_importances_

importances["mean"] = importances.mean(axis=1)
X_final = X_enc[importances.sort_values("mean", ascending=False).head(150).index]

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(weights)}

# ============================================================
# MÉTRICA CUSTOM: RECALL@3
# ============================================================
def recall_at_k(y_true, y_proba, k=3):
    topk = np.argsort(-y_proba, axis=1)[:, :k]
    return np.mean([y_true[i] in topk[i] for i in range(len(y_true))])

# ============================================================
# OPTUNA (MAXIMIZANDO RECALL@3)
# ============================================================
def objective(trial):

    n_layers = trial.suggest_int("n_layers", 1, 3)
    units = trial.suggest_categorical("units", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    l1 = trial.suggest_float("l1", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    inputs = keras.Input(shape=(X_train.shape[1],))
    x = inputs

    for i in range(n_layers):
        x = layers.Dense(units, activation="relu",
                         kernel_regularizer=regularizers.l1(l1))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        units = max(units // 2, 32)

    outputs = layers.Dense(len(np.unique(y)), activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=[keras.metrics.SparseTopKCategoricalAccuracy(k=3)]
    )

    model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=40,
        batch_size=batch_size,
        verbose=0,
        class_weight=class_weight,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )

    proba = model.predict(X_train, verbose=0)
    score = recall_at_k(y_train, proba, 3)

    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("\nBEST PARAMS:", study.best_params)

# ============================================================
# MODELO FINAL
# ============================================================
params = study.best_params

inputs = keras.Input(shape=(X_train.shape[1],))
x = inputs
units = params["units"]

for i in range(params["n_layers"]):
    x = layers.Dense(units, activation="relu",
                     kernel_regularizer=regularizers.l1(params["l1"]))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(params["dropout"])(x)
    units = max(units // 2, 32)

outputs = layers.Dense(len(np.unique(y)), activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(params["lr"]),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=3)
    ]
)

model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=80,
    batch_size=params["batch_size"],
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
    verbose=1
)

# ============================================================
# EVALUACIÓN FINAL
# ============================================================
proba = model.predict(X_test)
preds = np.argmax(proba, axis=1)

print("\n=== RESULTADOS ===")
print("Accuracy:", accuracy_score(y_test, preds))
print("F1:", f1_score(y_test, preds, average="weighted"))
print("LogLoss:", log_loss(y_test, proba))
print("Recall@3:", recall_at_k(y_test, proba, 3))



In [ ]:
CAMIBAR TRAIN A TEST
score = recall_at_k(y_train, proba, 3)

In [ ]:
# MODELO HIBRIDO - ENSEMBLE
proba_xgb = model_xgb.predict_proba(X_test)
proba_nn  = model_nn.predict(X_test)

proba_final = 0.5 * proba_xgb + 0.5 * proba_nn

In [ ]:
# STACKING - PROBS XGB COMO FEATURES PARA NN
X_stack = np.hstack([X_train, proba_xgb_train])

In [ ]:
# FEATURE AUGMENTATION
X_train["xgb_score_prod1"] = proba_xgb[:,0]
X_train["xgb_score_prod2"] = proba_xgb[:,1]